## Multi-protocol CNN Raw CSI Experiments

This notebook runs the fixed-capacity CNN across block, LOVO, and cross-session protocols with one independent run per configured seed.

In [1]:
from __future__ import annotations

import pandas as pd
import torch

from utils.config import (
    ANCHOR_GROUPS,
    ARCHITECTURE,
    BANDS_TO_RUN,
    DATA_DIR,
    DEFAULT_CNN_PARAMS,
    DEFAULT_OVERLAP_SIZE,
    DEFAULT_WINDOW_SIZE,
    EXPECTED_ANCHORS,
    EXPECTED_SUBCARRIERS,
    SEEDS,
)
from utils.DL.dl_pipeline import (
    create_position_label_encoder,
    get_cache_path,
    get_results_path,
    prepare_dl_data,
    print_torch_environment,
    run_dl_experiments,
    show_dl_results,
)


### Configuration

In [ ]:
CALIBRATION_MODE = "rssi"    # ("none", "packet_norm", "rssi")

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "normalization": "empty_baseline",  # none | zscore | minmax | packet_minmax | empty_baseline
    "epsilon": 1e-8,
}
NORMALIZATION_BASELINE_SCOPE = "per_session"

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": DEFAULT_WINDOW_SIZE,
    "overlap_size": DEFAULT_OVERLAP_SIZE,
    "require_all_esps": False,
}

BLOCK_COUNT = 10
TEST_SIZE = 0.30
VALIDATION_SIZE = 0.15
RANDOM_STATE = 42
FORCE_RETRAIN = False
SPLIT_MODES = ("block", "lovo")

CNN_PARAMS = {
    **DEFAULT_CNN_PARAMS,
    "model_label": "CNN",
    "epochs": 70,
    "patience": 15,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "validation_size": VALIDATION_SIZE,
    "n_blocks": BLOCK_COUNT,
    "anchor_groups": ANCHOR_GROUPS,
    "torch_version": torch.__version__,
}

preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
preproc_opts["baseline_scope"] = NORMALIZATION_BASELINE_SCOPE
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)

feature_cache_dir = get_cache_path(preproc_opts, feat_opts)
results_dir = get_results_path()
plots_dir = results_dir / "plots"
for directory in (plots_dir, results_dir / "predictions", results_dir / "tables"):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Feature cache path: {feature_cache_dir}")
print(f"Results path: {results_dir}")


Feature cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-none/feat=win60-step0
Results path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results


### Environment

In [3]:
DEVICE = print_torch_environment(require_cuda=True)


torch.__version__: 2.11.0+cu128
torch.version.cuda: 12.8
torch.cuda.get_device_name(0): NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
torch.cuda.get_device_capability(0): (12, 0)
CUDA matmul smoke test result: [[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0], [8.0, 9.0, 10.0, 11.0], [12.0, 13.0, 14.0, 15.0]] (PASS)


### Data

In [4]:
processed_magnitude_data, feature_dataframes, csv_diagnostics, magnitude_summary = prepare_dl_data(
    DATA_DIR,
    calibration_mode=CALIBRATION_MODE,
    csv_options=CSV_PROCESSING_OPTIONS,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
)
display(magnitude_summary.head())


Scenarios present: 1
Locations: 53 | Users: 6 | ESPs: 19
[cache hit] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-none/feat=win60-step0/2_4ghz.parquet
[cache hit] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-none/feat=win60-step0/5ghz.parquet
[cache hit] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-none/feat=win60-step0/fusion.parquet


,scenario,location,user,esp,trial,samples,subcarriers,normalization,baseline_scope
0,1,C-1,06,16,01,2088,56,none,
1,1,C-1,06,15,01,1942,56,none,
2,1,C-1,06,09,01,1689,50,none,
3,1,C-1,06,07,01,1965,50,none,
4,1,C-1,06,04,01,1913,50,none,


In [5]:
for band in BANDS_TO_RUN:
    df = feature_dataframes[band]
    print(f"{band}: {df.shape[0]} windows, {df.shape[1]} columns")
    print(f"{band} dataframe hash: {pd.util.hash_pandas_object(df, index=True).sum()}")


2.4 GHz: 13588 windows, 2708 columns
2.4 GHz dataframe hash: 12901719627776270762
5 GHz: 14478 windows, 3368 columns
5 GHz dataframe hash: 4746810533186240626
Fusion: 13568 windows, 6068 columns
Fusion dataframe hash: 10770637710876130155


In [6]:
label_encoder = create_position_label_encoder(
    feature_dataframes,
    results_dir=results_dir,
    expected_classes=52,
)
print(label_encoder.classes_)


[CNN] label classes saved to /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/manifests/cnn_label_classes.json
['A-1' 'A-13' 'A-14' 'A-2' 'A-5' 'B-1' 'B-10' 'B-11' 'B-12' 'B-13' 'B-14'
 'B-2' 'B-5' 'B-8' 'C-1' 'C-10' 'C-11' 'C-14' 'C-2' 'C-3' 'C-4' 'C-5'
 'C-6' 'C-7' 'C-8' 'C-9' 'D-1' 'D-2' 'D-3' 'D-4' 'D-5' 'D-6' 'D-7' 'D-8'
 'E-1' 'E-10' 'E-11' 'E-12' 'E-13' 'E-2' 'E-3' 'E-4' 'E-5' 'E-6' 'E-7'
 'E-8' 'F-10' 'F-11' 'F-12' 'F-13' 'F-5' 'F-8']


### Train And Evaluate

In [7]:
cnn_runs = run_dl_experiments(
    processed_magnitude_data,
    feature_dataframes,
    bands=BANDS_TO_RUN,
    split_modes=SPLIT_MODES,
    label_encoder=label_encoder,
    device=DEVICE,
    results_dir=results_dir,
    plots_dir=plots_dir,
    params=CNN_PARAMS,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    expected_subcarriers=EXPECTED_SUBCARRIERS,
    expected_anchors=EXPECTED_ANCHORS,
    architecture=ARCHITECTURE,
    seeds=SEEDS,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    n_blocks=BLOCK_COUNT,
    val_size=VALIDATION_SIZE,
    force_retrain=FORCE_RETRAIN,
)



=== 2.4 GHz ===
[window arrays] cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-none/feat=win60-step0/window_arrays/2_4ghz
[window arrays] 2.4 GHz: shape=(13588, 9, 50, 60), dtype=float16
[window arrays] 2.4 GHz anchors: ['esp_01', 'esp_02', 'esp_03', 'esp_04', 'esp_05', 'esp_07', 'esp_08', 'esp_09', 'esp_10']
2.4 GHz: anchors=9, subcarriers=50, windows=13588
WINDOW IDENTITY PASS: 2.4 GHz (13588 windows)
[trial filter] split=block trials=['01'] kept=8906/13588
[protocol] split=block trials_used=['01'] n_train=4947 n_test=1392 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '02', '03', '04', '05', '06'] test_users=['01', '02', '03', '04', '05', '06']
[trial filter] split=block trials=['01'] kept=8906/13588
[protocol] split=block trials_used=['01'] n_train=4947 n_test=1392 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '02', '03', '04', '05', '06'] test_users=['01', '02', '03', '04', '05', '06']
SPLIT IDENTITY 

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[CNN] 2.4 GHz/block fold=single epoch 01/70: train_loss=3.9517 train_acc=0.0219 val_loss=3.9561 val_acc=0.0133 seconds=2.8
[CNN] 2.4 GHz/block fold=single epoch 02/70: train_loss=3.9246 train_acc=0.0448 val_loss=3.9158 val_acc=0.0467 seconds=0.2
[CNN] 2.4 GHz/block fold=single epoch 03/70: train_loss=3.8795 train_acc=0.0617 val_loss=3.8619 val_acc=0.0467 seconds=0.2
[CNN] 2.4 GHz/block fold=single epoch 04/70: train_loss=3.7768 train_acc=0.0674 val_loss=3.9595 val_acc=0.0200 seconds=0.2
[CNN] 2.4 GHz/block fold=single epoch 05/70: train_loss=3.6241 train_acc=0.0873 val_loss=3.6483 val_acc=0.0800 seconds=0.2
[CNN] 2.4 GHz/block fold=single epoch 06/70: train_loss=3.4931 train_acc=0.1052 val_loss=3.7341 val_acc=0.0467 seconds=0.2
[CNN] 2.4 GHz/block fold=single epoch 07/70: train_loss=3.3754 train_acc=0.1171 val_loss=3.5176 val_acc=0.1267 seconds=0.2
[CNN] 2.4 GHz/block fold=single epoch 08/70: train_loss=3.2694 train_acc=0.1397 val_loss=3.5005 val_acc=0.1200 seconds=0.2
[CNN] 2.4 GHz/bl

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn__2_4ghz__block__none__s42__a8e3cb.parquet
[plots] saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/dl__cnn__2_4ghz__block__none__s42__a8e3cb/training_curves.png at dpi=200. PNG is raster; set PLOT_FORMAT='pdf' for a LaTeX vector figure without other code changes.
[DL GPU] run_id=dl__cnn__2_4ghz__block__none__s42__a8e3cb peak_cuda_memory_bytes=514714624
[results upsert] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/runs.csv: 71 row(s)
Seeds: random=43, numpy=43, torch=43
[trial filter] split=block trials=['01'] kept=8906/13588
[protocol] split=block trials_used=['01'] n_train=4947 n_test=1392 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '02', '03', '04', '05', '06'] test_users=['01', '02', '03', '04', '05', '06']
[trial filter] split=block trials=['01'] kept=4947/4947
[protocol] split=block trials_used=['01'] n_train=3014 n_test=150

...

[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn__fusion__block__none__s43__89f1a2.parquet
[plots] saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/dl__cnn__fusion__block__none__s43__89f1a2/training_curves.png at dpi=200. PNG is raster; set PLOT_FORMAT='pdf' for a LaTeX vector figure without other code changes.
[DL GPU] run_id=dl__cnn__fusion__block__none__s43__89f1a2 peak_cuda_memory_bytes=978418688
[results upsert] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/runs.csv: 78 row(s)
Seeds: random=44, numpy=44, torch=44
[trial filter] split=block trials=['01'] kept=8889/13568
[protocol] split=block trials_used=['01'] n_train=4934 n_test=1388 users=['01', '02', '03', '04', '05', '06'] train_users=['01', '02', '03', '04', '05', '06'] test_users=['01', '02', '03', '04', '05', '06']
[trial filter] split=block trials=['01'] kept=4934/4934
[protocol] split=block trials_used=['01'] n_train=3003 n_test=150

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[CNN] Fusion/block fold=single epoch 01/70: train_loss=3.9389 train_acc=0.0246 val_loss=3.8995 val_acc=0.0533 seconds=2.3
[CNN] Fusion/block fold=single epoch 02/70: train_loss=3.8453 train_acc=0.0776 val_loss=3.7608 val_acc=0.0600 seconds=0.4
[CNN] Fusion/block fold=single epoch 03/70: train_loss=3.6640 train_acc=0.0932 val_loss=3.6726 val_acc=0.0867 seconds=0.4
[CNN] Fusion/block fold=single epoch 04/70: train_loss=3.4436 train_acc=0.1375 val_loss=3.5036 val_acc=0.1533 seconds=0.4
[CNN] Fusion/block fold=single epoch 05/70: train_loss=3.2554 train_acc=0.1612 val_loss=3.3146 val_acc=0.1133 seconds=0.4
[CNN] Fusion/block fold=single epoch 06/70: train_loss=3.0794 train_acc=0.1785 val_loss=3.1399 val_acc=0.1400 seconds=0.4
[CNN] Fusion/block fold=single epoch 07/70: train_loss=2.8596 train_acc=0.2401 val_loss=3.4676 val_acc=0.1000 seconds=0.3
[CNN] Fusion/block fold=single epoch 08/70: train_loss=2.6990 train_acc=0.2604 val_loss=3.5592 val_acc=0.1333 seconds=0.4
[CNN] Fusion/block fold=

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn__fusion__block__none__s44__d87d56.parquet
[plots] saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/dl__cnn__fusion__block__none__s44__d87d56/training_curves.png at dpi=200. PNG is raster; set PLOT_FORMAT='pdf' for a LaTeX vector figure without other code changes.
[DL GPU] run_id=dl__cnn__fusion__block__none__s44__d87d56 peak_cuda_memory_bytes=978418688
[results upsert] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/runs.csv: 79 row(s)
[DL seeds] mean +/- std across seeds written to tables/seed_summary; LOVO uses each seed's fold mean and keeps within-seed fold std separate.


In [8]:
cnn_summary, test_accuracy_comparison = show_dl_results(results_dir)
display(cnn_summary)

display(test_accuracy_comparison)


,run_id,timestamp,family,model,band,split,seed,normalization,baseline_scope,window_size,...,p90_distance_error_max,samples_mean,samples_std,samples_min,samples_max,majority_position_accuracy_min,majority_position_accuracy_max,majority_room_accuracy_min,majority_room_accuracy_max,n_test_windows
3,dl__cnn__2_4ghz__block__ebl-session__s42__df21ec,2026-07-22T18:36:31.182108+00:00,dl,cnn,2_4ghz,block,42.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,dl__cnn__2_4ghz__block__ebl-session__s43__210c56,2026-07-22T18:36:47.242239+00:00,dl,cnn,2_4ghz,block,43.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,dl__cnn__2_4ghz__block__ebl-session__s44__86f94f,2026-07-22T18:37:03.908150+00:00,dl,cnn,2_4ghz,block,44.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,dl__cnn__5ghz__block__ebl-session__s42__df21ec,2026-07-22T18:37:26.695344+00:00,dl,cnn,5ghz,block,42.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,dl__cnn__5ghz__block__ebl-session__s43__210c56,2026-07-22T18:37:44.903111+00:00,dl,cnn,5ghz,block,43.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,dl__cnn__5ghz__block__ebl-session__s44__86f94f,2026-07-22T18:38:04.402553+00:00,dl,cnn,5ghz,block,44.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,dl__cnn__fusion__block__ebl-session__s42__df21ec,2026-07-22T18:38:41.941273+00:00,dl,cnn,fusion,block,42.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,dl__cnn__fusion__block__ebl-session__s43__210c56,2026-07-22T18:39:07.169167+00:00,dl,cnn,fusion,block,43.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,dl__cnn__fusion__block__ebl-session__s44__86f94f,2026-07-22T18:39:27.186644+00:00,dl,cnn,fusion,block,44.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,dl__cnn__2_4ghz__block__ebl-session__s42__eb2083,2026-07-22T22:25:24.462285+00:00,dl,cnn,2_4ghz,block,42.0,empty_baseline,session,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,band,model,seed,position_accuracy,parameter_count
3,2_4ghz,cnn,42.0,0.248563,76020.0
12,2_4ghz,cnn,42.0,0.286101,76020.0
61,2_4ghz,cnn,42.0,0.082927,76020.0
70,2_4ghz,cnn,42.0,0.307471,76020.0
4,2_4ghz,cnn,43.0,0.316092,76020.0
13,2_4ghz,cnn,43.0,0.276036,76020.0
62,2_4ghz,cnn,43.0,0.100000,76020.0
71,2_4ghz,cnn,43.0,0.272989,76020.0
5,2_4ghz,cnn,44.0,0.320402,76020.0
14,2_4ghz,cnn,44.0,0.293457,76020.0
